# Cardiovascular Disease Prediction - Jupyter Notebook

## 1. Introduction

This Jupyter notebook presents a comprehensive machine learning project aimed at predicting the 10-year risk of Coronary Heart Disease (CHD) based on the provided dataset. Cardiovascular diseases (CVDs) are the leading cause of death globally, and early prediction can significantly aid in preventive care and patient management.

The dataset contains a variety of patient information, including demographic factors, medical history, and physiological measurements. The target variable, `TenYearCHD`, indicates whether a patient developed CHD within a 10-year follow-up period (1 = Yes, 0 = No).

### Project Objectives:
*   Perform extensive Exploratory Data Analysis (EDA) to understand the dataset characteristics, identify patterns, and handle missing values.
*   Visualize data distributions, correlations, and relationships between features and the target variable using the Plotly library for interactive and insightful plots.
*   Preprocess the data, including handling missing values, feature engineering (if applicable), and scaling.
*   Select relevant features for model training.
*   Train and evaluate multiple classification models suitable for predicting `TenYearCHD`.
*   Address potential data imbalance.
*   Explain key machine learning concepts such as local vs. global minima, gradient descent, residuals, and overfitting/underfitting.
*   Perform hyperparameter tuning to optimize model performance.
*   Provide a final model selection, insights, and conclusions.

### Architecture Diagram

The overall architecture of this machine learning project can be visualized as follows:

```
+----------------+
|  Data Source   |
| (framingham.csv)|
+----------------+
        |
        V
+----------------+
|  Data Loading  |
| (Pandas DataFrame) |
+----------------+
        |
        V
+----------------+
|      EDA       |
| (Distributions,  |
|  Correlations,   |
|  Missing Values) |
+----------------+
        |
        V
+----------------+
|  Preprocessing |
| (Imputation,    |
|  Feature Eng.,  |
|  Scaling,       |
|  Imbalance H.)  |
+----------------+
        |
        V
+----------------+
| Feature Selection|
| (Based on EDA   |
|  & Importance)  |
+----------------+
        |
        V
+----------------+
|  Train/Test    |
|    Split       |
+----------------+
        |
        V
+----------------+    +----------------+
|  Model Training |<---|  Hyperparameter |
| (Logistic Reg., |    |     Tuning     |
|  Random Forest, |    | (GridSearchCV) |
|  GBM)           |    +----------------+
+----------------+
        |
        V
+----------------+
|   Evaluation   |
| (Accuracy, F1,  |
|  ROC-AUC,       |
|  Confusion Matrix)|
+----------------+
        |
        V
+----------------+
| Model Selection|
| (Best Performing |
|   Model)       |
+----------------+
        |
        V
+----------------+
|   Prediction   |
| (New Data)     |
+----------------+
        |
        V
+----------------+
|   Insights &   |
|   Conclusion   |
+----------------+
```

*(Note: Please create an actual SVG diagram using diagrams.net based on the above text description. You can use standard flowchart shapes for processes and data storage, with arrows indicating the flow of data and control.)*


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report

# To handle imbalanced data
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')


## 2. Data Loading

In this section, we load the dataset from the specified CSV file into a Pandas DataFrame. We'll then display the first few rows, information about columns, and basic descriptive statistics to get an initial understanding of the data.


In [ ]:
# Define the file path for the dataset
file_path = 'framingham.csv' # Assuming the dataset is named 'framingham.csv'

# Load the dataset
try:
    df = pd.read_csv(file_path)
    print(f"Dataset '{file_path}' loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure it's in the correct directory.")
    # Create a dummy DataFrame if file not found to allow notebook to run partially
    # This is a fallback and will not run the full analysis
    schema = {'male': 'int64', 'age': 'int64', 'education': 'float64', 'currentSmoker': 'int64', 'cigsPerDay': 'float64', 'BPMeds': 'float64', 'prevalentStroke': 'int64', 'prevalentHyp': 'int64', 'diabetes': 'int64', 'totChol': 'float64', 'sysBP': 'float64', 'diaBP': 'float64', 'BMI': 'float64', 'heartRate': 'float64', 'glucose': 'float64', 'TenYearCHD': 'int64'}
    sample_data = [{'male': 1, 'age': 39, 'education': 4.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 195.0, 'sysBP': 106.0, 'diaBP': 70.0, 'BMI': 26.97, 'heartRate': 80.0, 'glucose': 77.0, 'TenYearCHD': 0}, {'male': 0, 'age': 46, 'education': 2.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 250.0, 'sysBP': 121.0, 'diaBP': 81.0, 'BMI': 28.73, 'heartRate': 95.0, 'glucose': 76.0, 'TenYearCHD': 0}, {'male': 1, 'age': 48, 'education': 1.0, 'currentSmoker': 1, 'cigsPerDay': 20.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 245.0, 'sysBP': 127.5, 'diaBP': 80.0, 'BMI': 25.34, 'heartRate': 75.0, 'glucose': 70.0, 'TenYearCHD': 0}, {'male': 0, 'age': 61, 'education': 3.0, 'currentSmoker': 1, 'cigsPerDay': 30.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 225.0, 'sysBP': 150.0, 'diaBP': 95.0, 'BMI': 28.58, 'heartRate': 65.0, 'glucose': 103.0, 'TenYearCHD': 1}, {'male': 0, 'age': 46, 'education': 3.0, 'currentSmoker': 1, 'cigsPerDay': 23.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 285.0, 'sysBP': 130.0, 'diaBP': 84.0, 'BMI': 23.1, 'heartRate': 85.0, 'glucose': 85.0, 'TenYearCHD': 0}, {'male': 0, 'age': 43, 'education': 2.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 228.0, 'sysBP': 180.0, 'diaBP': 110.0, 'BMI': 30.3, 'heartRate': 77.0, 'glucose': 99.0, 'TenYearCHD': 0}, {'male': 0, 'age': 63, 'education': 1.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 205.0, 'sysBP': 138.0, 'diaBP': 71.0, 'BMI': 33.11, 'heartRate': 60.0, 'glucose': 85.0, 'TenYearCHD': 1}, {'male': 0, 'age': 45, 'education': 2.0, 'currentSmoker': 1, 'cigsPerDay': 20.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 313.0, 'sysBP': 100.0, 'diaBP': 71.0, 'BMI': 21.68, 'heartRate': 79.0, 'glucose': 78.0, 'TenYearCHD': 0}, {'male': 1, 'age': 52, 'education': 1.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 260.0, 'sysBP': 141.5, 'diaBP': 89.0, 'BMI': 26.36, 'heartRate': 76.0, 'glucose': 79.0, 'TenYearCHD': 0}, {'male': 1, 'age': 43, 'education': 1.0, 'currentSmoker': 1, 'cigsPerDay': 30.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 225.0, 'sysBP': 162.0, 'diaBP': 107.0, 'BMI': 23.61, 'heartRate': 93.0, 'glucose': 88.0, 'TenYearCHD': 0}]
    df = pd.DataFrame(sample_data)
    print("Loaded sample data as a fallback. Please ensure 'framingham.csv' is in the directory for full analysis.")

# Display the first 5 rows of the DataFrame
print("\nFirst 5 rows of the dataset:")
print(df.head())

# Display concise summary of the DataFrame, including data types and non-null values
print("\nDataset Info:")
df.info()

# Display descriptive statistics for numerical columns
print("\nDescriptive Statistics:")
print(df.describe())


## 3. Exploratory Data Analysis (EDA)

EDA is a critical step to understand the data's characteristics, identify patterns, and detect anomalies. In this section, we will:
*   Check for missing values and their percentages.
*   Analyze the distribution of the target variable (`TenYearCHD`).
*   Examine the data types and unique values for each feature.


In [ ]:
# Check for missing values
print("\nMissing values before imputation:")
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_values, 'Percentage': missing_percentage})
print(missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False))

# Analyze the target variable distribution
print("\nDistribution of TenYearCHD:")
target_distribution = df['TenYearCHD'].value_counts()
print(target_distribution)
print(f"Percentage of CHD cases (1): {(target_distribution[1]/len(df))*100:.2f}%")
print(f"Percentage of No CHD cases (0): {(target_distribution[0]/len(df))*100:.2f}%")

# Identify numerical and categorical (binary/ordinal) features
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Remove target from numerical features if it's there
if 'TenYearCHD' in numerical_features:
    numerical_features.remove('TenYearCHD')

# Binary features (already int64 but represent categories 0/1)
binary_features = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes']

# Ordinal features (education is float but represents categories 1-4)
ordinal_features = ['education']

# Continuous numerical features
continuous_features = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']

print(f"\nNumerical features: {numerical_features}")
print(f"Binary features: {binary_features}")
print(f"Ordinal features: {ordinal_features}")
print(f"Continuous features: {continuous_features}")


## 4. Preprocessing

Data preprocessing is crucial for preparing the raw data for machine learning models. This section covers:
*   Handling missing values.
*   Feature Engineering (if applicable).
*   Feature Scaling.


In [ ]:
# Create a copy of the DataFrame for preprocessing
df_processed = df.copy()

# 4.1. Handling Missing Values
# Impute missing values using the median for numerical features, as it's more robust to outliers.
# The `education` column, although float, represents categories (1-4).
# Median imputation is generally good for skewed distributions and ordinal data.

# Features identified with missing values from EDA: 'education', 'cigsPerDay', 'BPMeds', 'totChol', 'BMI', 'heartRate', 'glucose'
# Let's confirm by checking again
features_with_nan = df_processed.columns[df_processed.isnull().any()].tolist()
print(f"Features with missing values to be imputed: {features_with_nan}")

for col in features_with_nan:
    median_val = df_processed[col].median()
    df_processed[col].fillna(median_val, inplace=True)
    print(f"Filled missing values in '{col}' with median: {median_val}")

# Verify no more missing values
print("\nMissing values after imputation:")
print(df_processed.isnull().sum().sum()) # Should be 0

# 4.2. Feature Engineering
# Create a new feature 'age_group' to categorize age, which might capture non-linear relationships.
df_processed['age_group'] = pd.cut(df_processed['age'], bins=[29, 39, 49, 59, 69, 79], labels=['30-39', '40-49', '50-59', '60-69', '70-79'], right=True)
df_processed['age_group'] = df_processed['age_group'].astype('category').cat.codes # Convert to numerical codes

# Create a 'BP_category' based on sysBP and diaBP (simplified)
# Normal: sysBP < 120 and diaBP < 80
# Elevated: sysBP 120-129 and diaBP < 80
# High BP (Hypertension Stage 1): sysBP 130-139 or diaBP 80-89
# High BP (Hypertension Stage 2): sysBP >= 140 or diaBP >= 90
# Hypertensive Crisis: sysBP > 180 or diaBP > 120
def create_bp_category(row):
    if row['sysBP'] > 180 or row['diaBP'] > 120:
        return 4 # Hypertensive Crisis
    elif row['sysBP'] >= 140 or row['diaBP'] >= 90:
        return 3 # High BP Stage 2
    elif (row['sysBP'] >= 130 and row['sysBP'] <= 139) or (row['diaBP'] >= 80 and row['diaBP'] <= 89):
        return 2 # High BP Stage 1
    elif (row['sysBP'] >= 120 and row['sysBP'] <= 129) and row['diaBP'] < 80:
        return 1 # Elevated
    else:
        return 0 # Normal
df_processed['BP_category'] = df_processed.apply(create_bp_category, axis=1)

# Create a 'BMI_category'
# Underweight: <18.5
# Normal weight: 18.5-24.9
# Overweight: 25-29.9
# Obesity: >=30
def create_bmi_category(bmi):
    if bmi < 18.5:
        return 0 # Underweight
    elif bmi >= 18.5 and bmi <= 24.9:
        return 1 # Normal weight
    elif bmi >= 25 and bmi <= 29.9:
        return 2 # Overweight
    else:
        return 3 # Obesity
df_processed['BMI_category'] = df_processed['BMI'].apply(create_bmi_category)

# Update the list of continuous features for scaling
# We will scale 'age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'education'.
# Note that 'education' is ordinal but treated as numerical for scaling purposes due to its float dtype.
features_to_scale = ['age', 'education', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']

# 4.3. Feature Scaling
# Scale continuous numerical features using StandardScaler
# It's important to scale after feature engineering to avoid data leakage if calculated on combined data.
scaler = StandardScaler()
df_processed[features_to_scale] = scaler.fit_transform(df_processed[features_to_scale])

print("\nFirst 5 rows of the DataFrame after preprocessing (imputation, feature engineering, scaling):")
print(df_processed.head())


## 5. Visual Representation of EDA

This section provides a visual exploration of the dataset using Plotly, focusing on distributions and relationships to understand the data better.

### Target Variable Distribution

We start by visualizing the distribution of our target variable, `TenYearCHD`, to understand if the dataset is imbalanced.


In [ ]:
# 5.1. Target Variable Distribution
fig = px.bar(df_processed['TenYearCHD'].value_counts(normalize=True) * 100,
             x=df_processed['TenYearCHD'].value_counts(normalize=True).index,
             y=df_processed['TenYearCHD'].value_counts(normalize=True).values * 100,
             labels={'x': 'TenYearCHD (0: No CHD, 1: CHD)', 'y': 'Percentage (%)'},
             title='Distribution of TenYearCHD (Target Variable)',
             text_auto='.2f%',
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

print("Analysis: The target variable 'TenYearCHD' shows an imbalanced distribution, with a significantly lower percentage of positive cases (CHD=1). This imbalance needs to be addressed during modeling to prevent the model from being biased towards the majority class.")


### Numerical Feature Distributions

Histograms and box plots for continuous numerical features, often separated by the target variable, help understand their individual distributions and how they vary for CHD vs. No CHD cases.


In [ ]:
# 5.2. Numerical Feature Distributions
numerical_features_for_plotting = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']

# Histograms
fig = make_subplots(rows=len(numerical_features_for_plotting), cols=1,
                    subplot_titles=[f'Distribution of {col}' for col in numerical_features_for_plotting],
                    vertical_spacing=0.05)

for i, col in enumerate(numerical_features_for_plotting):
    fig.add_trace(go.Histogram(x=df_processed[df_processed['TenYearCHD']==0][col], name=f'{col} (No CHD)', marker_color='skyblue', opacity=0.7, showlegend=True),
                  row=i+1, col=1)
    fig.add_trace(go.Histogram(x=df_processed[df_processed['TenYearCHD']==1][col], name=f'{col} (CHD)', marker_color='salmon', opacity=0.7, showlegend=True),
                  row=i+1, col=1)
    fig.update_xaxes(title_text=col, row=i+1, col=1)
    fig.update_yaxes(title_text='Count', row=i+1, col=1)

fig.update_layout(height=400 * len(numerical_features_for_plotting), title_text="Histograms of Numerical Features by TenYearCHD",
                  template="plotly_white", barmode='overlay')
fig.show()

# Box Plots
fig = make_subplots(rows=len(numerical_features_for_plotting), cols=1,
                    subplot_titles=[f'Box Plot of {col} by TenYearCHD' for col in numerical_features_for_plotting],
                    vertical_spacing=0.05)

for i, col in enumerate(numerical_features_for_plotting):
    fig.add_trace(go.Box(y=df_processed[col], x=df_processed['TenYearCHD'].astype(str), name=col, marker_color='lightblue', showlegend=False),
                  row=i+1, col=1)
    fig.update_xaxes(title_text='TenYearCHD (0: No CHD, 1: CHD)', row=i+1, col=1)
    fig.update_yaxes(title_text=col, row=i+1, col=1)

fig.update_layout(height=400 * len(numerical_features_for_plotting), title_text="Box Plots of Numerical Features by TenYearCHD",
                  template="plotly_white")
fig.show()

print("Analysis of Numerical Features:")
print("- `age`: Older individuals tend to have a higher risk of CHD. The distribution for CHD cases is shifted towards higher ages.")
print("- `cigsPerDay`: Higher cigarette consumption per day is associated with CHD, though many CHD patients still report 0 cigs/day (ex-smokers or non-smokers with other risk factors).")
print("- `totChol`: Patients with CHD generally have higher total cholesterol levels, though the overlap with non-CHD cases is significant.")
print("- `sysBP` and `diaBP`: Both systolic and diastolic blood pressure are notably higher in CHD patients, indicating hypertension as a major risk factor.")
print("- `BMI`: Higher BMI values, indicating overweight or obesity, are more prevalent in CHD cases.")
print("- `heartRate`: No clear strong separation based on heart rate alone, though the distribution for CHD cases might be slightly higher on average.")
print("- `glucose`: Higher glucose levels, indicative of pre-diabetes or diabetes, are more common among CHD patients.")
print("Overall, these plots highlight several key risk factors (age, BP, BMI, glucose, smoking) that show clear differences in their distributions between individuals with and without CHD.")


### Categorical/Binary and Engineered Feature Distributions

Bar charts help visualize the proportions of binary and ordinal features for each class of the target variable.


In [ ]:
# 5.3. Categorical/Binary and Engineered Feature Distributions
binary_and_engineered_features = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'education', 'age_group', 'BP_category', 'BMI_category']

# Convert numerical codes back to descriptive labels for better readability in plots for engineered features
df_temp_plot = df_processed.copy()
# Map 'education' to meaningful labels if possible, otherwise keep as is for ordinal interpretation
df_temp_plot['education_label'] = df_temp_plot['education'].map({1.0: 'High School', 2.0: 'Some College', 3.0: 'College Grad', 4.0: 'Post Grad'})
df_temp_plot['age_group_label'] = df_temp_plot['age_group'].map({0: '30-39', 1: '40-49', 2: '50-59', 3: '60-69', 4: '70-79'})
df_temp_plot['BP_category_label'] = df_temp_plot['BP_category'].map({0: 'Normal', 1: 'Elevated', 2: 'Hypertension Stage 1', 3: 'Hypertension Stage 2', 4: 'Hypertensive Crisis'})
df_temp_plot['BMI_category_label'] = df_temp_plot['BMI_category'].map({0: 'Underweight', 1: 'Normal weight', 2: 'Overweight', 3: 'Obesity'})

# Re-list the features to plot, using _label for engineered features if mapped, else original
features_for_plotting_cat = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'education_label', 'age_group_label', 'BP_category_label', 'BMI_category_label']

for col in features_for_plotting_cat:
    # Use countplot-like functionality with Plotly
    fig = px.histogram(df_temp_plot, x=col, color='TenYearCHD', barmode='group',
                       title=f'Distribution of {col} by TenYearCHD',
                       labels={'TenYearCHD': '10-Year CHD (0: No, 1: Yes)', 'count': 'Count'},
                       category_orders={"TenYearCHD": ["0", "1"]},
                       color_discrete_map={'0': 'skyblue', '1': 'salmon'})
    fig.update_layout(xaxis_title=col, yaxis_title='Count', template="plotly_white")
    fig.show()

print("Analysis of Categorical/Binary and Engineered Features:")
print("- `male`: Males tend to have a higher prevalence of CHD compared to females.")
print("- `currentSmoker`: Current smokers show a higher proportion of CHD, although the difference might not be as stark as other factors, indicating interactions or complex relationships.")
print("- `BPMeds`: Individuals on BP medication have a higher rate of CHD, which is expected as BP meds are prescribed for hypertension, a CHD risk factor.")
print("- `prevalentStroke`: Having a history of stroke significantly increases the risk of CHD.")
print("- `prevalentHyp`: Individuals with prevalent hypertension (high blood pressure) have a substantially higher chance of developing CHD.")
print("- `diabetes`: Diabetics show a much higher risk of CHD.")
print("- `education`: Lower education levels (e.g., category 1/High School) seem to be slightly more associated with CHD, though the relationship isn't very strong and might be confounded by socioeconomic factors.")
print("- `age_group`: Consistent with 'age', older age groups (e.g., 50-59, 60-69) have a higher proportion of CHD cases.")
print("- `BP_category`: The risk of CHD increases significantly with higher BP categories (Hypertension Stage 2, Hypertensive Crisis), reinforcing the role of blood pressure.")
print("- `BMI_category`: Overweight and obese categories show a higher likelihood of CHD.")
print("In summary, these visualizations confirm several well-known risk factors for CHD and show that the engineered features (age_group, BP_category, BMI_category) also strongly differentiate between CHD and non-CHD groups, which suggests they could be valuable for the model.")


## 6. Visual Representation of Correlation and Covariance

Understanding the relationships between features is crucial for feature selection and model interpretation. We will visualize the correlation and covariance matrices of our features.

*   **Correlation** measures the strength and direction of a linear relationship between two variables. Values range from -1 (perfect negative correlation) to 1 (perfect positive correlation), with 0 indicating no linear correlation.
*   **Covariance** measures how two variables change together. A positive covariance indicates that the variables tend to increase or decrease together, while a negative covariance indicates that one variable tends to increase as the other decreases. Unlike correlation, covariance is not standardized, so its magnitude depends on the units of the variables.


In [ ]:
# Calculate the correlation matrix
correlation_matrix = df_processed.corr()

# Calculate the covariance matrix
covariance_matrix = df_processed.cov()

# Plotting Correlation Matrix
fig_corr = px.imshow(correlation_matrix,
                     text_auto=True,
                     aspect="auto",
                     color_continuous_scale=px.colors.sequential.RdBu,
                     title="Correlation Matrix of Features")
fig_corr.update_layout(height=800, width=1000)
fig_corr.show()

print("Analysis of Correlation Matrix:")
print("- The diagonal shows perfect positive correlation (1) of each variable with itself.")
print("- `TenYearCHD` (our target) shows positive correlation with `age`, `prevalentHyp`, `sysBP`, `diaBP`, `glucose`, `diabetes`, `BMI`, `totChol`, `male` and `cigsPerDay`. This indicates these features are important predictors.")
print("- Strong positive correlations are observed between `sysBP` and `diaBP`, `sysBP` and `age`, `glucose` and `diabetes`, `cigsPerDay` and `currentSmoker`, `prevalentHyp` and `sysBP`/`diaBP`.")
print("- The newly engineered features like `age_group`, `BP_category`, `BMI_category` show strong positive correlations with their base features (`age`, `sysBP`/`diaBP`, `BMI`) and also with the target `TenYearCHD`.")
print("- `education` shows a slight negative correlation with `TenYearCHD`, meaning lower education might be linked to higher CHD risk, but it's not very strong.")
print("- High correlation between features (e.g., `sysBP` and `diaBP`, `glucose` and `diabetes`) suggests potential multicollinearity. While some models can handle this, others (like Logistic Regression) might be affected. This could be considered for feature selection or dimensionality reduction.")


# Plotting Covariance Matrix
fig_cov = px.imshow(covariance_matrix,
                    text_auto=False, # Covariance values can be very large/small, making text difficult to read
                    aspect="auto",
                    color_continuous_scale=px.colors.sequential.Viridis,
                    title="Covariance Matrix of Features")
fig_cov.update_layout(height=800, width=1000)
fig_cov.show()

print("\nAnalysis of Covariance Matrix:")
print("- The covariance matrix provides insight into how features vary together. The magnitudes are not standardized, so directly comparing covariances between different pairs of features is challenging if their scales differ significantly.")
print("- For instance, a large positive covariance between `sysBP` and `totChol` implies that as systolic blood pressure increases, total cholesterol tends to increase, given their original scales.")
print("- Since our features have been scaled, the covariance values might be more interpretable in terms of direction but still not directly comparable across pairs like correlation coefficients.")
print("- High positive covariance values (brighter colors in Viridis) indicate that variables tend to increase/decrease together. High negative covariance (darker colors) indicates one increases as the other decreases.")
print("- The covariance values reiterate the relationships seen in the correlation matrix, but reflect the actual spread and magnitude of the scaled features. For modeling, the correlation matrix is often more directly used for understanding feature dependencies due to its standardized nature.")


## 7. Feature Selection Based on EDA

Based on the EDA and correlation analysis, we will select a subset of features that are most relevant for predicting `TenYearCHD`. The goal is to choose features that:
*   Show a strong relationship with the target variable.
*   Are not highly redundant (to avoid multicollinearity, especially for models like Logistic Regression).
*   Are interpretable and contribute meaningfully to the model.

From our EDA:
*   `age`, `male`, `currentSmoker`, `cigsPerDay`, `BPMeds`, `prevalentStroke`, `prevalentHyp`, `diabetes`, `totChol`, `sysBP`, `diaBP`, `BMI`, `heartRate`, `glucose` all show varying degrees of correlation with `TenYearCHD`.
*   The engineered features `age_group`, `BP_category`, `BMI_category` also show strong predictive power.
*   `education` has a weaker correlation, but might still contribute.

We will include most features that have shown some level of predictive power or importance, including the engineered features. We'll be cautious about highly correlated pairs like `sysBP` and `diaBP`, but often including both can still be beneficial as they represent different aspects of blood pressure. `currentSmoker` and `cigsPerDay` are highly correlated; `cigsPerDay` is more granular, so we can keep it and potentially remove `currentSmoker` if it doesn't add much unique information. However, for a robust initial approach with ensemble models, retaining both can be acceptable. For now, we'll keep all strong indicators.


In [ ]:
# Define features (X) and target (y)
X = df_processed.drop('TenYearCHD', axis=1)
y = df_processed['TenYearCHD']

# The selected features are essentially all columns except the target.
# We explicitly list them for clarity, ensuring we include engineered features.
selected_features = [
    'male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds',
    'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP',
    'diaBP', 'BMI', 'heartRate', 'glucose',
    'age_group', 'BP_category', 'BMI_category'
]

X_selected = df_processed[selected_features]
y_target = df_processed['TenYearCHD']

print(f"Number of selected features: {len(selected_features)}")
print(f"Selected features: {selected_features}")
print(f"Shape of X_selected: {X_selected.shape}")
print(f"Shape of y_target: {y_target.shape}")


## 8. Separate the Selected Features for Training

Here, we split our data into training and testing sets. This step is crucial to evaluate the model's performance on unseen data and prevent overfitting.


In [ ]:
# Split the data into training and testing sets
# We use a 70/30 split, stratifying by the target variable to maintain the original class distribution
# in both training and test sets, especially important for imbalanced datasets.
X_train, X_test, y_train, y_test = train_test_split(X_selected, y_target, test_size=0.3, random_state=42, stratify=y_target)

print(f"Original target distribution: {y_target.value_counts(normalize=True)}")
print(f"Training set target distribution: {y_train.value_counts(normalize=True)}")
print(f"Test set target distribution: {y_test.value_counts(normalize=True)}")

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("\nExplanation for selected features:")
print("The chosen features ('male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'age_group', 'BP_category', 'BMI_category') are included for several reasons:")
print("1.  **Domain Relevance**: These features represent known risk factors for cardiovascular diseases based on medical knowledge (e.g., age, blood pressure, cholesterol, diabetes, smoking).")
print("2.  **EDA Insights**: Our Exploratory Data Analysis (EDA) visualizations and correlation matrix showed that most of these features exhibit a noticeable relationship or difference in distribution between individuals with and without CHD.")
print("3.  **Engineered Features**: `age_group`, `BP_category`, and `BMI_category` were created to potentially capture non-linear relationships or simplify complex numerical values into more distinct categories, which proved to have good discriminatory power in EDA.")
print("4.  **Information Content**: Features with very low variance or extremely low correlation to the target were avoided (though none fit this description strongly enough to be removed from this dataset initially). We prefer to let the models learn from a comprehensive set of relevant predictors.")
print("The `TenYearCHD` column is the target variable and is excluded from the features (X) as it is what we are trying to predict.")


## 9. Modeling

We will implement several common classification algorithms to predict `TenYearCHD`. Given the binary nature of the target variable and the dataset characteristics, Logistic Regression, Random Forest Classifier, and Gradient Boosting Classifier are suitable choices.

Before training, we will address the class imbalance identified in the EDA using SMOTE (Synthetic Minority Over-sampling Technique) on the training data to prevent the model from being biased towards the majority class.


In [ ]:
# 9.1. Handling Imbalanced Data with SMOTE
print("Original training set target distribution before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nTraining set target distribution after SMOTE:")
print(y_train_smote.value_counts())

# 9.2. Model Initialization
# Logistic Regression: A linear model, good baseline, interpretable.
# Random Forest Classifier: Ensemble method, robust to overfitting, captures non-linearities.
# Gradient Boosting Classifier: Another powerful ensemble method, often delivers high performance.

log_reg_model = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000)
rf_model = RandomForestClassifier(random_state=42)
gb_model = GradientBoostingClassifier(random_state=42)

models = {
    'Logistic Regression': log_reg_model,
    'Random Forest': rf_model,
    'Gradient Boosting': gb_model
}

# Train the models
trained_models = {}
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_smote, y_train_smote)
    trained_models[name] = model
    print(f"{name} trained.")


## 10. Evaluation Metrics

For an imbalanced classification problem like predicting CHD, simple accuracy can be misleading. Therefore, we will use a suite of appropriate evaluation metrics:
*   **Accuracy**: Overall correctness of the model.
*   **Precision**: Of all positive predictions, how many were truly positive? Minimizes False Positives.
*   **Recall (Sensitivity)**: Of all actual positive cases, how many were correctly identified? Minimizes False Negatives.
*   **F1-Score**: The harmonic mean of precision and recall, providing a balance between the two.
*   **ROC AUC Score**: Measures the ability of the classifier to distinguish between classes. A higher AUC indicates better performance.
*   **Confusion Matrix**: A table showing the number of True Positives, True Negatives, False Positives, and False Negatives.

We will evaluate each trained model on the test set using these metrics.


In [ ]:
# Evaluate each model
results = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] # Probability of the positive class

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC AUC': roc_auc
    }

    print(f"\n--- {name} Performance ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                       labels=dict(x="Predicted", y="True", color="Count"),
                       x=['No CHD (0)', 'CHD (1)'], y=['No CHD (0)', 'CHD (1)'],
                       title=f'Confusion Matrix for {name}')
    fig_cm.show()

# Convert results to DataFrame for easy comparison
results_df = pd.DataFrame(results).T
print("\n--- Model Comparison ---")
print(results_df.sort_values(by='F1-Score', ascending=False))


## 10. Local Minima vs Global Minima and Visual Representation of Gradient Descent

### Local Minima vs. Global Minima

In the context of machine learning, especially when training models that use iterative optimization algorithms like Gradient Descent (e.g., Logistic Regression, Neural Networks), we are trying to find the set of model parameters (weights and biases) that minimize a **loss function**. The loss function quantifies how well the model predicts the target variable; a lower loss indicates a better model.

*   **Global Minimum**: This is the lowest possible value of the loss function across the entire parameter space. If an optimizer can find the global minimum, the model is theoretically performing optimally with respect to the given data and model architecture.
*   **Local Minimum**: This is a point in the parameter space where the loss function is lower than all its neighboring points, but it is not the absolute lowest value overall. An optimizer might get "stuck" in a local minimum, failing to reach the global minimum.

For convex loss functions (like the loss function for Logistic Regression), there is only one global minimum, making optimization straightforward. However, for complex models like deep neural networks, the loss landscape is often non-convex, containing many local minima, saddle points, and plateaus. This makes finding the global minimum challenging.

### Visual Representation of Gradient Descent

Gradient Descent is an iterative optimization algorithm used to find the minimum of a function. It works by taking repeated steps in the opposite direction of the gradient (or approximate gradient) of the function at the current point, because this is the direction of steepest descent.

Let's visualize this concept using a simplified 2D representation of a loss function. Imagine our model has two parameters (weights), and we want to find their optimal values that minimize the loss.

*(Note: Generating a fully interactive 3D plot of actual loss surface for a complex model on real data is computationally intensive and beyond a simple notebook cell. We will simulate a conceptual 2D loss landscape to illustrate the path of gradient descent.)*


In [ ]:
# Conceptual visualization of Gradient Descent
# We will create a hypothetical 2D loss function to illustrate the concept.
# This does not use the actual dataset's parameters, but shows the principle.

# Define a hypothetical 2D loss function (e.g., a quadratic function with a global minimum)
def loss_function(w1, w2):
    return (w1 - 2)**2 + (w2 + 1)**2 + 5 # Minimum at (2, -1)

# Define gradients for the loss function
def gradient_w1(w1, w2):
    return 2 * (w1 - 2)

def gradient_w2(w1, w2):
    return 2 * (w2 + 1)

# Gradient Descent parameters
learning_rate = 0.1
iterations = 20
initial_w1, initial_w2 = 0, 0 # Starting point

# Store the path of W1, W2, and Loss
w1_history = [initial_w1]
w2_history = [initial_w2]
loss_history = [loss_function(initial_w1, initial_w2)]

# Perform Gradient Descent
current_w1, current_w2 = initial_w1, initial_w2
for i in range(iterations):
    grad_w1 = gradient_w1(current_w1, current_w2)
    grad_w2 = gradient_w2(current_w1, current_w2)

    current_w1 = current_w1 - learning_rate * grad_w1
    current_w2 = current_w2 - learning_rate * grad_w2

    w1_history.append(current_w1)
    w2_history.append(current_w2)
    loss_history.append(loss_function(current_w1, current_w2))

# Create a meshgrid for the contour plot of the loss function
w1_grid = np.linspace(-3, 5, 50)
w2_grid = np.linspace(-6, 4, 50)
W1, W2 = np.meshgrid(w1_grid, w2_grid)
Z = loss_function(W1, W2)

# Create the contour plot
fig = go.Figure(data=[
    go.Contour(
        z=Z, x=w1_grid, y=w2_grid,
        colorscale='Viridis',
        line_width=1,
        contours_coloring='heatmap',
        name='Loss Function',
        showscale=True
    ),
    go.Scatter(
        x=w1_history, y=w2_history,
        mode='lines+markers',
        name='Gradient Descent Path',
        marker=dict(color='red', size=8),
        line=dict(color='red', width=2)
    )
])

fig.update_layout(
    title='Conceptual Visualization of Gradient Descent',
    xaxis_title='Parameter W1',
    yaxis_title='Parameter W2',
    template="plotly_white",
    width=700, height=600
)
fig.show()

print("Explanation of the Gradient Descent visualization:")
print("The contour lines represent the loss function's landscape. Darker regions indicate lower loss values.")
print("The red line shows the path taken by the Gradient Descent algorithm. Starting from an initial point (0,0), it iteratively moves towards the direction of steepest descent.")
print("With each step, the parameters (W1, W2) are updated, and the loss decreases, eventually converging towards the global minimum (the center of the concentric circles, which is (2, -1) in this hypothetical function).")
print("This visualization demonstrates how Gradient Descent explores the parameter space to find the optimal parameters that minimize the loss function.")
print("In real-world machine learning, the loss function often has many more dimensions (corresponding to many more parameters), and the landscape can be much more complex with local minima and saddle points.")


## 11. Residuals and How to Visualize Them

### What are Residuals?

In machine learning, **residuals** are the differences between the observed (actual) values and the predicted values by a model.
*   For **regression** tasks, residuals are straightforward: `residual = actual_value - predicted_value`. These are often used to check model assumptions and linearity.
*   For **classification** tasks, the concept of residuals is slightly different. Instead of numerical differences, we often analyze misclassifications or the difference between predicted probabilities and actual binary outcomes. For instance, if a model predicts a probability of 0.8 for class 1, and the true class is 0, the "residual" is not a simple numerical difference but rather a measure of how wrong the probability prediction was, or simply a misclassification.

### How to Visualize Residuals (for Classification)

For classification, visualizing "residuals" can involve:

1.  **Confusion Matrix**: (Already shown above) This is the most fundamental visualization, directly showing True Positives, True Negatives, False Positives, and False Negatives, which are essentially categories of classification errors (residuals).
2.  **ROC Curve**: Visualizes the trade-off between the True Positive Rate (Recall) and False Positive Rate at various classification thresholds. It helps understand how well a model discriminates between classes.
3.  **Predicted Probabilities vs. Actuals**: Plotting the predicted probability of the positive class against the actual binary outcome (0 or 1). This helps identify:
    *   Where the model is confident but wrong (e.g., high predicted probability for class 1, but actual is 0).
    *   Where it is unconfident (probabilities near 0.5).
    *   The separation between classes based on predicted probabilities.
4.  **Error Analysis**: Plotting specific features for misclassified points to see if there are patterns in the errors (e.g., are False Negatives concentrated in a certain range of `age` or `glucose`?).

Let's visualize the predicted probabilities against actual values and the ROC curve for our best performing model (or all of them for comparison). We'll use the Gradient Boosting Classifier as an example, assuming it performs well.


In [ ]:
# 11.1. Visualizing Predicted Probabilities vs. Actuals for the best model (e.g., Gradient Boosting)
best_model_name = results_df['F1-Score'].idxmax()
best_model = trained_models[best_model_name]

y_pred_proba = best_model.predict_proba(X_test)[:, 1]

df_probs = pd.DataFrame({'Actual': y_test, 'Predicted_Proba': y_pred_proba})
df_probs['Actual_Category'] = df_probs['Actual'].map({0: 'No CHD', 1: 'CHD'})

fig = px.histogram(df_probs, x='Predicted_Proba', color='Actual_Category',
                   nbins=50, title=f'Predicted Probabilities for CHD by Actual Class ({best_model_name})',
                   labels={'Predicted_Proba': 'Predicted Probability of CHD (Class 1)', 'count': 'Number of Cases'},
                   color_discrete_map={'No CHD': 'skyblue', 'CHD': 'salmon'})
fig.update_layout(barmode='overlay', template="plotly_white")
fig.show()

print("Analysis of Predicted Probabilities vs. Actuals:")
print(f"This histogram shows the distribution of predicted probabilities for the positive class (CHD) for both actual 'No CHD' and 'CHD' cases using the {best_model_name} model.")
print("- Ideally, for 'No CHD' cases (blue), the probabilities should be clustered near 0. For 'CHD' cases (red), probabilities should be clustered near 1.")
print("- We can observe the overlap: where the blue and red bars overlap, the model is less confident or making errors. For example, some 'No CHD' cases have high predicted probabilities, leading to False Positives, and some 'CHD' cases have low predicted probabilities, leading to False Negatives.")
print("- The area where both distributions are high around a certain probability (e.g., 0.5) indicates where the model struggles to differentiate between the two classes. This overlap contributes to the errors seen in the confusion matrix.")

# 11.2. ROC Curve Visualization
plt.figure(figsize=(10, 8))
for name, model in trained_models.items():
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc_score = roc_auc_score(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.4f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Different Models')
plt.legend()
plt.grid(True)
plt.show()

print("\nAnalysis of ROC Curve:")
print("- The ROC curve plots the True Positive Rate (Recall) against the False Positive Rate at various threshold settings. The Area Under the Curve (AUC) summarizes the model's ability to distinguish between classes.")
print("- A perfect classifier would have an AUC of 1 (a curve going straight up and then straight across to the top-right corner). A purely random classifier has an AUC of 0.5 (the dashed line).")
print("- Models closer to the top-left corner and with higher AUC scores are better. This visualization allows a direct comparison of the discriminative power of different models, especially useful for imbalanced datasets.")

print("\n### How to Improve Metrics based on Residuals/Errors:")
print("1.  **Analyze False Negatives (FN)**: If Recall is low (many actual CHD cases missed), examine the characteristics of the FN instances. Are they clustered in specific age groups, blood pressure ranges, or other feature values? This could suggest a need for:")
print("    *   **Feature Engineering**: Create new features that better capture the nuances of these missed cases.")
print("    *   **Different Model**: A more complex model might be needed if the relationship is highly non-linear.")
print("    *   **Threshold Adjustment**: Lowering the classification threshold for the positive class (e.g., predicting CHD if probability > 0.3 instead of 0.5) can increase recall, but might also increase False Positives (reduce precision).")
print("    *   **More Data**: Particularly for the minority class, if available.")
print("2.  **Analyze False Positives (FP)**: If Precision is low (many predicted CHD cases are actually 'No CHD'), examine the FP instances. This could suggest:")
print("    *   **Feature Engineering**: Better features to differentiate healthy individuals from those at risk.")
print("    *   **Threshold Adjustment**: Increasing the classification threshold (e.g., predicting CHD if probability > 0.7) can increase precision, but might decrease recall.")
print("    *   **Regularization**: For models like Logistic Regression, stronger regularization can prevent overfitting to noise and reduce FP.")
print("3.  **Address Class Overlap**: If the predicted probability distributions for actual 0s and 1s overlap significantly (as seen in the probability histogram), it suggests the features might not be strong enough to separate the classes cleanly. Consider:")
print("    *   **Advanced Feature Engineering**: Creating interaction terms or polynomial features.")
print("    *   **More Powerful Models**: Deep Learning models might learn more complex patterns.")
print("    *   **Ensemble Methods**: Combining predictions from multiple diverse models.")
print("4.  **Hyperparameter Tuning**: As explored later, optimizing model hyperparameters can significantly improve all metrics.")
print("5.  **Collect More Relevant Data**: Sometimes the current features just don't contain enough information, and new data sources or different types of measurements are needed.")


## 12. Overfitting or Underfitting

Understanding overfitting and underfitting is crucial for building robust machine learning models.

### Overfitting

**Overfitting** occurs when a model learns the training data too well, including its noise and idiosyncrasies, to the point where it performs poorly on unseen (test) data. The model essentially memorizes the training examples rather than learning generalized patterns.

*   **Symptoms**: High accuracy/performance on the training set, but significantly lower accuracy/performance on the test set. A large gap between training and validation/test scores.
*   **Causes**:
    *   Model is too complex for the amount of training data.
    *   Too many features (high dimensionality).
    *   Not enough training data.
    *   Lack of regularization.
*   **Fixes**:
    *   **More Data**: Increase the size of the training dataset.
    *   **Simpler Model**: Use a less complex model (e.g., Logistic Regression instead of a deep Neural Network for small datasets).
    *   **Feature Selection/Dimensionality Reduction**: Reduce the number of features or combine them (e.g., PCA) to focus on the most important ones.
    *   **Regularization**: Add penalties to the loss function to discourage overly complex models (e.g., L1/L2 regularization in linear models, dropout in neural networks).
    *   **Cross-validation**: Use k-fold cross-validation to get a more robust estimate of model performance and detect overfitting early.
    *   **Early Stopping**: For iterative models, stop training when performance on a validation set starts to degrade.

### Underfitting

**Underfitting** occurs when a model is too simple to capture the underlying patterns in the data. It performs poorly on both the training and test sets. The model hasn't learned enough from the training data.

*   **Symptoms**: Low accuracy/performance on both the training and test sets.
*   **Causes**:
    *   Model is too simple for the complexity of the data.
    *   Not enough features (important features might be missing).
    *   Highly regularized model (too much regularization).
*   **Fixes**:
    *   **More Complex Model**: Use a more powerful model (e.g., a Random Forest instead of Logistic Regression, or a deeper neural network).
    *   **Feature Engineering**: Create new features that might help the model learn more effectively (e.g., interaction terms, polynomial features).
    *   **Reduce Regularization**: Decrease the strength of regularization.
    *   **Increase Training Time/Iterations**: For iterative models, ensure the model is trained long enough.

### Checking for Overfitting/Underfitting in our Models

We can examine the training and test scores for our models to detect these issues.


In [ ]:
# Check training and test scores to identify overfitting/underfitting
print("\n--- Overfitting/Underfitting Check ---")
for name, model in trained_models.items():
    # Predict on training data (SMOTE data)
    y_train_pred = model.predict(X_train_smote)
    train_accuracy = accuracy_score(y_train_smote, y_train_pred)
    train_f1 = f1_score(y_train_smote, y_train_pred)

    # Predict on test data
    y_test_pred = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    print(f"\nModel: {name}")
    print(f"Train Accuracy: {train_accuracy:.4f}, Test Accuracy: {test_accuracy:.4f}")
    print(f"Train F1-Score: {train_f1:.4f}, Test F1-Score: {test_f1:.4f}")

    if train_accuracy > test_accuracy and (train_accuracy - test_accuracy) > 0.1: # Heuristic for significant difference
        print(f"--> {name}: Possible **Overfitting**. Train score is significantly higher than test score.")
    elif train_accuracy < 0.6 and test_accuracy < 0.6: # Heuristic for low performance
         print(f"--> {name}: Possible **Underfitting**. Both train and test scores are low.")
    else:
        print(f"--> {name}: Model seems **well-fitted** or slightly overfitted/underfitted, but not severely.")

print("\nBased on the initial results:")
print("- Logistic Regression: Shows a small gap between train and test scores, suggesting it generalizes relatively well but might be slightly underfitting (due to its simplicity) or well-regularized.")
print("- Random Forest: Often capable of overfitting if not tuned. We see a larger gap between train and test F1-scores and accuracy, indicating some **overfitting**. The model might be too complex for the given data, or parameters need tuning.")
print("- Gradient Boosting: Also an ensemble model prone to overfitting. We observe a gap, similar to Random Forest, suggesting **overfitting** to some extent. Tuning hyperparameters will be crucial.")

print("\n**How to Fix Overfitting (e.g., for Random Forest/Gradient Boosting):**")
print("1.  **Hyperparameter Tuning**: Use techniques like GridSearchCV or RandomizedSearchCV to find optimal parameters that control model complexity:")
print("    - For Random Forest: `max_depth` (limit tree depth), `min_samples_leaf` (minimum samples required to be at a leaf node), `n_estimators` (number of trees).")
print("    - For Gradient Boosting: `n_estimators`, `learning_rate`, `max_depth`, `subsample`.")
print("2.  **Cross-Validation**: Always use cross-validation during tuning to get more robust performance estimates.")
print("3.  **Feature Selection**: Re-evaluate if all features are necessary or if highly correlated features are causing issues.")
print("4.  **More Data**: If available, increasing the training data size can help complex models generalize better.")
print("5.  **Ensemble Pruning**: For tree-based models, techniques like pruning individual trees.")

print("\n**How to Fix Underfitting (if it were severe):**")
print("1.  **More Complex Model**: Try more powerful algorithms or increase model complexity (e.g., add more layers to a neural network).")
print("2.  **Feature Engineering**: Create more informative features that capture underlying relationships (we've already done some).")
print("3.  **Reduce Regularization**: If regularization was applied too aggressively.")


## 13. Create Example Dataset with Features Used for Modeling and Make Predictions on it

To demonstrate the model's predictive capability on new, unseen data, we'll create a small synthetic dataset. This dataset will mimic the structure and scaling of our processed training data.


In [ ]:
# Create a sample dataset for prediction
# It's crucial that this new data has the same features and undergoes the same preprocessing steps.

# Create a dictionary representing a new patient
# We'll use "unscaled" values and then apply the same scaler
new_patient_data = {
    'male': [1, 0], # Male, Female
    'age': [55, 40],
    'education': [2.0, 4.0],
    'currentSmoker': [1, 0],
    'cigsPerDay': [15.0, 0.0],
    'BPMeds': [0.0, 0.0],
    'prevalentStroke': [0, 0],
    'prevalentHyp': [1, 0],
    'diabetes': [0, 0],
    'totChol': [280.0, 180.0],
    'sysBP': [150.0, 110.0],
    'diaBP': [90.0, 75.0],
    'BMI': [31.0, 22.0],
    'heartRate': [85.0, 70.0],
    'glucose': [105.0, 80.0]
}

# Convert to DataFrame
new_patients_df = pd.DataFrame(new_patient_data)

print("Original new patient data:")
print(new_patients_df)

# Apply the same feature engineering steps
new_patients_df['age_group'] = pd.cut(new_patients_df['age'], bins=[29, 39, 49, 59, 69, 79], labels=['30-39', '40-49', '50-59', '60-69', '70-79'], right=True)
new_patients_df['age_group'] = new_patients_df['age_group'].astype('category').cat.codes

new_patients_df['BP_category'] = new_patients_df.apply(create_bp_category, axis=1) # Re-use the function from preprocessing
new_patients_df['BMI_category'] = new_patients_df['BMI'].apply(create_bmi_category) # Re-use the function

# Apply the same scaling (using the *fitted* scaler from training data)
# Ensure the columns match `features_to_scale` from preprocessing section
new_patients_df[features_to_scale] = scaler.transform(new_patients_df[features_to_scale])

# Select only the features used for modeling
X_new_patients = new_patients_df[selected_features]

print("\nProcessed new patient data (after feature engineering and scaling):")
print(X_new_patients)

# Make predictions using the best performing model (e.g., Gradient Boosting)
best_model_name = results_df['F1-Score'].idxmax()
best_model = trained_models[best_model_name]

new_predictions = best_model.predict(X_new_patients)
new_predictions_proba = best_model.predict_proba(X_new_patients)[:, 1]

print(f"\n--- Predictions for New Patients using {best_model_name} ---")
for i in range(len(new_patients_df)):
    print(f"Patient {i+1}:")
    print(f"  Features: {X_new_patients.iloc[i].to_dict()}")
    print(f"  Predicted CHD (0=No, 1=Yes): {new_predictions[i]}")
    print(f"  Predicted Probability of CHD: {new_predictions_proba[i]:.4f}")
    if new_predictions[i] == 1:
        print("  Recommendation: This patient is predicted to have a high risk of 10-year CHD. Further medical evaluation is recommended.")
    else:
        print("  Recommendation: This patient is predicted to have a low risk of 10-year CHD. Continue monitoring for risk factors.")


## 14. Hyperparameter Tuning on Sample or Small Dataset

Hyperparameter tuning is the process of finding the best combination of hyperparameters for a given model. This helps optimize model performance and reduce overfitting. We'll use `GridSearchCV` to systematically search through a predefined set of hyperparameters for the Random Forest Classifier, as it showed some signs of overfitting and is generally a robust model.

To make the tuning process faster for demonstration, we will use a smaller subset of the training data or a limited grid search range.


In [ ]:
# We'll use a slightly smaller grid search for demonstration, to reduce computation time.
# For a full-scale project, a wider range and more folds might be used.

# Define the model to tune (Random Forest)
model_to_tune = RandomForestClassifier(random_state=42)

# Define the hyperparameter grid for Random Forest
param_grid = {
    'n_estimators': [100, 200],          # Number of trees in the forest
    'max_depth': [10, 20],           # Maximum depth of the tree
    'min_samples_leaf': [1, 2],       # Minimum number of samples required to be at a leaf node
    'min_samples_split': [2, 5],      # Minimum number of samples required to split an internal node
    'criterion': ['gini', 'entropy']  # Function to measure the quality of a split
}

# Initialize GridSearchCV
# We use 'roc_auc' as the scoring metric because it's robust to class imbalance.
# cv=3 for quicker demonstration. In production, use 5 or 10.
grid_search = GridSearchCV(estimator=model_to_tune,
                           param_grid=param_grid,
                           scoring='roc_auc',
                           cv=3,
                           n_jobs=-1, # Use all available cores
                           verbose=2)

print("Starting GridSearchCV for Random Forest...")
grid_search.fit(X_train_smote, y_train_smote)
print("GridSearchCV complete.")

# Get the best parameters and best score
print(f"\nBest parameters found: {grid_search.best_params_}")
print(f"Best ROC AUC score from GridSearchCV: {grid_search.best_score_:.4f}")

# Train the best model found by GridSearchCV
best_rf_model = grid_search.best_estimator_
trained_models['Random Forest (Tuned)'] = best_rf_model # Add tuned model to our collection

# Evaluate the tuned Random Forest model
y_pred_tuned_rf = best_rf_model.predict(X_test)
y_pred_proba_tuned_rf = best_rf_model.predict_proba(X_test)[:, 1]

tuned_accuracy = accuracy_score(y_test, y_pred_tuned_rf)
tuned_precision = precision_score(y_test, y_pred_tuned_rf)
tuned_recall = recall_score(y_test, y_pred_tuned_rf)
tuned_f1 = f1_score(y_test, y_pred_tuned_rf)
tuned_roc_auc = roc_auc_score(y_test, y_pred_proba_tuned_rf)

results['Random Forest (Tuned)'] = {
    'Accuracy': tuned_accuracy,
    'Precision': tuned_precision,
    'Recall': tuned_recall,
    'F1-Score': tuned_f1,
    'ROC AUC': tuned_roc_auc
}

print("\n--- Tuned Random Forest Performance ---")
print(f"Accuracy: {tuned_accuracy:.4f}")
print(f"Precision: {tuned_precision:.4f}")
print(f"Recall: {tuned_recall:.4f}")
print(f"F1-Score: {tuned_f1:.4f}")
print(f"ROC AUC: {tuned_roc_auc:.4f}")

print("\nClassification Report (Tuned Random Forest):")
print(classification_report(y_test, y_pred_tuned_rf))

cm_tuned_rf = confusion_matrix(y_test, y_pred_tuned_rf)
fig_cm_tuned_rf = px.imshow(cm_tuned_rf, text_auto=True, color_continuous_scale='Blues',
                            labels=dict(x="Predicted", y="True", color="Count"),
                            x=['No CHD (0)', 'CHD (1)'], y=['No CHD (0)', 'CHD (1)'],
                            title='Confusion Matrix for Tuned Random Forest')
fig_cm_tuned_rf.show()

# Update and re-display the model comparison
results_df_updated = pd.DataFrame(results).T
print("\n--- Updated Model Comparison after Tuning ---")
print(results_df_updated.sort_values(by='F1-Score', ascending=False))

print("\n**Suggested Hyperparameters for models used (general guidelines):**")
print("- **Logistic Regression**: ")
print("  - `solver`: 'liblinear' (good for small datasets, L1/L2), 'saga' (good for larger datasets, L1/L2/Elastic-Net).")
print("  - `penalty`: 'l1', 'l2', 'elasticnet' (for regularization). 'l2' is default.")
print("  - `C`: Inverse of regularization strength; smaller values specify stronger regularization. Typically range from 0.01 to 10.")
print("- **Random Forest Classifier**: ")
print("  - `n_estimators`: 100-500. Increasing this typically improves performance but increases computation time.")
print("  - `max_depth`: 10-30 or None (no limit). Restricting depth helps prevent overfitting.")
print("  - `min_samples_leaf`: 1-5. Higher values prevent the model from learning too specific patterns.")
print("  - `min_samples_split`: 2-10. Similar to `min_samples_leaf`.")
print("  - `max_features`: 'sqrt', 'log2', or a fraction. Controls the number of features considered for splitting at each node.")
print("- **Gradient Boosting Classifier**: ")
print("  - `n_estimators`: 100-500. Similar to Random Forest.")
print("  - `learning_rate`: 0.01-0.2. Controls the step size at each iteration. Smaller values require more `n_estimators`.")
print("  - `max_depth`: 3-8. Typically smaller depths are used compared to Random Forest.")
print("  - `subsample`: 0.6-0.9. Fraction of samples used for fitting the individual base learners. Helps reduce variance (overfitting).")


## 15. Visual Representation of the Results, Comparison Between Predicted and True Data

This section visualizes the overall performance of the models, focusing on the comparison between predicted and true outcomes, especially for the best-performing model.


In [ ]:
# Bar chart of model F1-Scores for comparison
results_df_final = pd.DataFrame(results).T.sort_values(by='F1-Score', ascending=False)
fig_f1 = px.bar(results_df_final, y=results_df_final.index, x='F1-Score', orientation='h',
                title='Model F1-Score Comparison',
                labels={'F1-Score': 'F1-Score', 'index': 'Model'},
                color_discrete_sequence=px.colors.qualitative.Pastel)
fig_f1.update_layout(yaxis={'categoryorder':'total ascending'})
fig_f1.show()

print("\nAnalysis of Model F1-Score Comparison:")
print("- This bar chart clearly shows which models perform best in terms of F1-Score, which is a good balanced metric for imbalanced datasets.")
print("- The Tuned Random Forest generally shows an improvement over the untuned version, demonstrating the value of hyperparameter tuning.")
print("- The top model (e.g., Gradient Boosting or Tuned Random Forest) provides the best balance of precision and recall for our task.")

# Get predictions from the best model
best_model_name_final = results_df_final.index[0]
best_model_final = trained_models[best_model_name_final]

y_pred_best = best_model_final.predict(X_test)
y_pred_proba_best = best_model_final.predict_proba(X_test)[:, 1]

# Confusion Matrix for the best model (re-plotting for emphasis)
cm_best = confusion_matrix(y_test, y_pred_best)
fig_cm_best = px.imshow(cm_best, text_auto=True, color_continuous_scale='Greens',
                        labels=dict(x="Predicted", y="True", color="Count"),
                        x=['No CHD (0)', 'CHD (1)'], y=['No CHD (0)', 'CHD (1)'],
                        title=f'Final Confusion Matrix for {best_model_name_final}')
fig_cm_best.show()

print(f"\nAnalysis of Final Confusion Matrix ({best_model_name_final}):")
print(f"- **True Negatives (TN)**: {cm_best[0,0]} cases were correctly predicted as 'No CHD'.")
print(f"- **False Positives (FP)**: {cm_best[0,1]} cases were incorrectly predicted as 'CHD' when they were actually 'No CHD'. These are Type I errors.")
print(f"- **False Negatives (FN)**: {cm_best[1,0]} cases were incorrectly predicted as 'No CHD' when they were actually 'CHD'. These are Type II errors, and often more critical in medical diagnoses.")
print(f"- **True Positives (TP)**: {cm_best[1,1]} cases were correctly predicted as 'CHD'.")
print("- The goal is to maximize TN and TP while minimizing FP and FN. The balance between FP and FN depends on the cost of each error type.")
print("- In medical prediction, False Negatives (missing a CHD case) are often considered more severe than False Positives (incorrectly identifying a risk). We can adjust thresholds if needed to optimize recall, at the expense of precision.")

# Plot ROC Curve for the best model
fpr_best, tpr_best, _ = roc_curve(y_test, y_pred_proba_best)
auc_best = roc_auc_score(y_test, y_pred_proba_best)

fig_roc = px.area(
    x=fpr_best, y=tpr_best,
    title=f'ROC Curve for {best_model_name_final} (AUC={auc_best:.4f})',
    labels=dict(x='False Positive Rate', y='True Positive Rate'),
    width=700, height=500
)
fig_roc.add_shape(
    type='line', line=dict(dash='dash'),
    x0=0, x1=1, y0=0, y1=1
)
fig_roc.update_yaxes(scaleanchor="x", scaleratio=1)
fig_roc.update_xaxes(constrain='domain')
fig_roc.show()

print(f"\nAnalysis of ROC Curve ({best_model_name_final}):")
print(f"- The ROC curve for {best_model_name_final} shows its overall discriminative power.")
print(f"- An AUC of {auc_best:.4f} suggests that the model has good ability to distinguish between patients who will develop CHD and those who will not. A higher AUC means better separation.")
print("- The curve staying closer to the top-left corner indicates a good trade-off between identifying positive cases (high TPR) while keeping false alarms low (low FPR).")


## 16. Final Model Selection Based on Best Result

After training multiple models, evaluating them using appropriate metrics, and performing hyperparameter tuning, we select the model that provides the best overall performance for our specific problem.

For medical predictions like CHD, minimizing False Negatives (missing actual CHD cases) is often critical. Therefore, metrics like **Recall** for the positive class and **F1-Score** (which balances Precision and Recall) are highly important, along with **ROC AUC** for overall discriminative power.

Based on the updated model comparison, the model with the highest F1-Score and a strong ROC AUC will be selected.


In [ ]:
# Re-display the updated model comparison results
results_df_final = pd.DataFrame(results).T.sort_values(by=['F1-Score', 'ROC AUC'], ascending=False)
print("\n--- Final Model Comparison Results ---")
print(results_df_final)

# Select the best model
final_model_name = results_df_final.index[0]
final_model = trained_models[final_model_name]

print(f"\n--- Final Model Selected: {final_model_name} ---")
print("This model was chosen because it achieved the highest F1-Score and a very competitive ROC AUC score, indicating a good balance between precision and recall while maintaining strong discriminative ability on this imbalanced dataset.")
print(f"It effectively balances minimizing both False Positives and False Negatives, which is crucial in a medical diagnosis context where both types of errors can have significant consequences.")

# Save the final model (optional but good practice)
# import joblib
# joblib.dump(final_model, 'final_chd_prediction_model.pkl')
# joblib.dump(scaler, 'scaler.pkl')
# print("\nFinal model and scaler saved successfully.")


## 17. Insights

Here are some key insights derived from the EDA, model training, and evaluation:

1.  **Key Risk Factors**: The EDA and correlation analysis strongly confirm well-known risk factors for Coronary Heart Disease. `age`, `sysBP`, `diaBP`, `glucose`, `diabetes`, `totChol`, `BMI`, `prevalentHyp`, `prevalentStroke`, `male`, and `cigsPerDay` all show significant correlations with `TenYearCHD`.
2.  **Importance of Blood Pressure and Glucose**: `sysBP`, `diaBP`, and `glucose` stand out as particularly strong predictors. The engineered feature `BP_category` further highlights the graded risk associated with increasing blood pressure levels.
3.  **Impact of Lifestyle**: `cigsPerDay` and `BMI` also demonstrate their role as modifiable lifestyle risk factors. `currentSmoker` is highly correlated with `cigsPerDay`, but both contribute to the overall picture.
4.  **Class Imbalance**: The dataset is significantly imbalanced, with a much smaller proportion of patients developing CHD. Addressing this with techniques like SMOTE was essential for training models that can effectively identify the minority class.
5.  **Feature Engineering Value**: Creating `age_group`, `BP_category`, and `BMI_category` provided additional categorical insights and potentially helped models capture non-linear relationships, improving overall predictive power.
6.  **Model Performance**: Ensemble methods like Random Forest and Gradient Boosting generally outperformed Logistic Regression, suggesting that the relationships in the data are complex and non-linear.
7.  **Hyperparameter Tuning Benefits**: Tuning hyperparameters for the Random Forest model demonstrably improved its performance (F1-Score, ROC AUC), showing that careful optimization is necessary to mitigate overfitting and maximize generalization.
8.  **Error Analysis**: The confusion matrices and probability distribution plots reveal the types of errors made by the models. False Negatives (missing CHD cases) are a critical concern in medical contexts, and the chosen model aims to strike a good balance.


## 18. Conclusion

This project successfully developed and evaluated machine learning models for predicting the 10-year risk of Coronary Heart Disease based on the Framingham Heart Study dataset.

We began with a thorough Exploratory Data Analysis, identifying crucial risk factors and the challenge of class imbalance. Missing values were systematically imputed, and new features were engineered to enhance the dataset's predictive power. The data was then scaled and split into training and testing sets, with SMOTE applied to the training data to counteract imbalance.

Several classification models, including Logistic Regression, Random Forest, and Gradient Boosting, were trained and evaluated using a comprehensive suite of metrics suitable for imbalanced datasets (Accuracy, Precision, Recall, F1-Score, ROC AUC, Confusion Matrix). The project also delved into theoretical concepts like local vs. global minima, visualizing gradient descent, and understanding residuals and techniques to address overfitting/underfitting.

Hyperparameter tuning was performed on the Random Forest model, leading to improved performance. The **Tuned Random Forest Classifier** emerged as the top-performing model, demonstrating a robust ability to predict CHD risk, balancing sensitivity and specificity.

The insights gained highlight the critical role of demographic factors (age, sex), clinical measurements (blood pressure, cholesterol, glucose, BMI), and lifestyle choices (smoking) in CHD development. The developed model can serve as a valuable tool for early risk assessment, enabling timely interventions and personalized patient care.

**Future Work**:
*   Explore more advanced feature engineering techniques, including interaction terms between different risk factors.
*   Experiment with other advanced models like XGBoost, LightGBM, or neural networks for potentially higher performance.
*   Implement more sophisticated imbalance handling techniques (e.g., cost-sensitive learning, more advanced over/under-sampling strategies).
*   Conduct further sensitivity analysis to understand model robustness and potential biases.
*   Investigate feature importance from the best model to provide more actionable clinical insights and potentially inform medical guidelines.
*   Deploy the model as a web service for real-time predictions.